# Day 5: Async fetcher (solution)

Reference solutions. Every cell is runnable; `practice.ipynb` is the blank-page twin.

## `s1_data.py`: Data — deque frontier, seen-set, heap (crawler state).

In [ ]:
import heapq
from collections import deque
graph = {"a": ["b", "c"], "b": ["c"], "c": []}
frontier, seen, order = deque(["a"]), {"a"}, []
while frontier:
    n = frontier.popleft()
    order.append(n)
    for m in graph[n]:
        if m not in seen:
            seen.add(m)
            frontier.append(m)

In [ ]:
assert order == ["a", "b", "c"]

In [ ]:
h = []
heapq.heappush(h, (2, "b")); heapq.heappush(h, (1, "a"))


In [ ]:
import heapq

assert heapq.heappop(h) == (1, "a")

**Then, vary it:**

- VARIATION: pop() from the right instead; BFS becomes DFS
- VARIATION: use heapq.nsmallest(2, ...) on a list of (priority, url)

## `s2_flow.py`: Flow — async generator, async comprehension.

In [ ]:
import asyncio
async def agen(n):
    for i in range(n):
        await asyncio.sleep(0)
        yield i

In [ ]:
import asyncio

async def main():
    assert [i async for i in agen(3)] == [0, 1, 2]
    assert [x * 2 async for x in agen(3) if x] == [2, 4]
await main()

**Then, vary it:**

- VARIATION: use `async for` in a plain loop and break early
- VARIATION: try `for i in agen(3)` and read the error

## `s3_functions.py`: Functions — async def, gather, async decorator.

In [ ]:
import asyncio, functools, time
def timed(fn):
    @functools.wraps(fn)
    async def w(*a, **k):
        t = time.perf_counter()
        r = await fn(*a, **k)
        w.last = time.perf_counter() - t
        return r
    return w
@timed
async def work(x):
    await asyncio.sleep(0.05)
    return x

In [ ]:
import asyncio, functools, time

async def main():
    assert await asyncio.gather(*(work(i) for i in range(5))) == [0, 1, 2, 3, 4]
    assert work.last < 0.2   # 5 x 0.05s ran concurrently, not 0.25s
await main()

**Then, vary it:**

- VARIATION: call work(1) without await; what do you get back?
- VARIATION: replace gather with a sequential for-await loop and time it

## `s4_classes.py`: Classes — async context manager.

In [ ]:
import asyncio
class Session:
    def __init__(self): self.open = False
    async def __aenter__(self):
        self.open = True
        return self
    async def __aexit__(self, *exc): self.open = False

In [ ]:
import asyncio

async def main():
    async with Session() as s:
        assert s.open
    assert not s.open
await main()

**Then, vary it:**

- VARIATION: build the same with @contextlib.asynccontextmanager
- VARIATION: raise inside the block; confirm __aexit__ still runs

## `s5_errors.py`: Errors — gather(return_exceptions), timeout, TaskGroup.

In [ ]:
import asyncio
async def boom(): raise ValueError("x")
async def slow(): await asyncio.sleep(1)

In [ ]:
import asyncio

async def main():
    r = await asyncio.gather(boom(), asyncio.sleep(0, "ok"), return_exceptions=True)
    assert isinstance(r[0], ValueError) and r[1] == "ok"
    try: await asyncio.wait_for(slow(), 0.05)
    except TimeoutError: pass
    else: raise AssertionError
    try:
        async with asyncio.TaskGroup() as tg:
            tg.create_task(boom())
    except* ValueError:
        pass
await main()

**Then, vary it:**

- VARIATION: use `async with asyncio.timeout(0.05):` instead of wait_for
- VARIATION: cancel a task and catch CancelledError (re-raise it!)

## `s6_stdlib.py`: Stdlib — batched, accumulate, reduce, statistics, seeded random.

In [ ]:
import operator, random, statistics
from functools import reduce
from itertools import accumulate, batched, chain
assert list(batched(range(5), 2)) == [(0, 1), (2, 3), (4,)]
assert list(accumulate([1, 2, 3])) == [1, 3, 6]
assert reduce(operator.mul, [1, 2, 3, 4]) == 24
assert statistics.median([3, 1, 2]) == 2 and statistics.mean([1, 2, 3]) == 2
assert random.Random(0).random() == random.Random(0).random()
assert list(chain([1], [2, 3])) == [1, 2, 3]

**Then, vary it:**

- VARIATION: batch URLs in groups of 3 for a rate-limited fetch
- VARIATION: shuffle a list with a seeded Random and sample 2

## `s7_concurrency.py`: Concurrency — Queue + worker tasks + Semaphore.

In [ ]:
import asyncio
async def main():
    q, out, sem = asyncio.Queue(), [], asyncio.Semaphore(2)
    async def worker():
        while True:
            n = await q.get()
            async with sem:
                await asyncio.sleep(0)
                out.append(n * n)
            q.task_done()
    ws = [asyncio.create_task(worker()) for _ in range(3)]
    for i in range(5): q.put_nowait(i)
    await q.join()
    for w in ws: w.cancel()
    assert sorted(out) == [0, 1, 4, 9, 16]
await main()

**Then, vary it:**

- VARIATION: add a None sentinel per worker for clean shutdown instead of cancel()
- VARIATION: use asyncio.to_thread to call a blocking function from a worker

## `s8_quality.py`: Quality — test async code with AsyncMock.

In [ ]:
import asyncio
from unittest.mock import AsyncMock
async def fetch_all(client, urls):
    return await asyncio.gather(*(client.get(u) for u in urls))
c = AsyncMock()
c.get.side_effect = lambda u: u.upper()

In [ ]:
import asyncio

assert await fetch_all(c, ['a', 'b']) == ['A', 'B']
assert c.get.await_count == 2

**Then, vary it:**

- VARIATION: make get raise for "b" and assert the error propagates
- VARIATION: assert call args with c.get.assert_any_await("a")

## `kata.py`: KATA — bounded-concurrency fetcher (fake I/O).

In [ ]:
import asyncio
async def fetch_all(urls, limit=2):
    sem, active, peak = asyncio.Semaphore(limit), 0, 0
    async def one(u):
        nonlocal active, peak
        async with sem:
            active += 1
            peak = max(peak, active)
            await asyncio.sleep(0.01)
            active -= 1
            return u.upper()
    return await asyncio.gather(*map(one, urls)), peak
res, peak = await fetch_all(list('abcde'))

In [ ]:
assert res == list("ABCDE") and peak == 2

**Then, vary it:**

- KATA: add a per-request timeout and return None for timed-out urls